# Sudoku con Qwen3.5 en Google Colab

En Colab, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**, y ejecuta las celdas en orden. La disponibilidad de GPU depende de Colab. El repositorio debe incluir este notebook y los cambios de `qwen.py` en la rama seleccionada.

Al reiniciar el entorno se pierden el repositorio clonado y los pesos descargados; vuelve a ejecutar las celdas. La primera descarga puede tardar varios minutos.


In [ ]:
import subprocess
import sys

# Cambia la URL o la rama si el equipo trabaja en otro repositorio o rama.
REPO_URL = "https://github.com/danvillablanca/harness-iagen-sudoku.git"
BRANCH = "main"  # Rama que contiene los cambios; modifica si corresponde.
REPO_DIR = "/content/harness-iagen-sudoku"

from pathlib import Path
if Path(REPO_DIR).exists():
    raise RuntimeError(f"{REPO_DIR} ya existe. Usa un entorno nuevo o elige otra ruta; no se sobrescribirá.")
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR], check=True)


## Dependencias

Se conserva el PyTorch y torchvision del entorno de Colab para aprovechar su instalación CUDA; se instalan las demás dependencias fijadas por el proyecto. Si cambias el tipo de GPU después de ejecutar esta celda, reinicia el entorno y comienza de nuevo.


In [ ]:
from pathlib import Path
requirements = Path(REPO_DIR, "requirements.txt").read_text().splitlines()
packages = [line.strip() for line in requirements
            if line.strip() and not line.lstrip().startswith("#")
            and not line.strip().lower().startswith(("torch==", "torchvision=="))]
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)


In [ ]:
import torch
import psutil
from transformers import Qwen3_5ForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible. Selecciona GPU T4 en el entorno de ejecución y vuelve a ejecutar las celdas.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CPU lógicas:", psutil.cpu_count(logical=True))
print("RAM disponible (GiB):", round(psutil.virtual_memory().available / 1024**3, 1))
print("VRAM libre (GiB):", round(torch.cuda.mem_get_info()[0] / 1024**3, 1))


## Ejecutar el experimento

El límite de salida incluye los tokens de razonamiento. Una respuesta incorrecta aparecerá como `Result: FAIL`; eso es un resultado del experimento, no un error de instalación.


In [ ]:
import sys
sys.path.insert(0, str(Path(REPO_DIR) / "Laboratorio" / "Qwen3.5"))
from config import ModelConfig
from models import load_model
from prompts import prompts
from timing import timed
from evaluation import evaluate
from display import print_result

THINKING = True
MAX_OUTPUT_TOKENS = 8192  # Reducir si la GPU se queda sin memoria o demora demasiado.
config = ModelConfig(backend="cuda", thinking=THINKING, max_output_tokens=MAX_OUTPUT_TOKENS)
case = prompts["sudoku9x9"]["hard"]
model, load_seconds = timed(load_model, config)
result, generation_seconds = timed(model.generate_with_stats, case.prompt)
report = evaluate(result.text, case)
print_result(case, report, result, config.max_output_tokens, load_seconds, generation_seconds)
